# 아이솔레이션 포레스트 테스트 (초안)

| 항목 | 내용 |
|---|---|
| 데이터 | `data/processed/minute/` (1분 요약, 장비 20대) |
| 목적 | 정상만으로 학습한 이상 탐지가 고장 24시간 전 구간을 구분하는지 확인 |
| 상태 | 초안 (기준선 확인용, 튜닝 없음) |

## 1. 방법

- **라벨**: 1분 요약의 TTF 3개 중 하나라도 86,400초(24시간) 이하이면 고장 전(1), 아니면 정상(0).
- **학습**: 학습 장비의 정상 분만 사용(무작위 30만 개). 고장 전 분은 학습에 쓰지 않는다.
- **입력**: 센서 16개의 1분 평균. 결측은 학습 데이터 중앙값으로 채움.
- **점수**: `-score_samples` (클수록 이상).
- **검증**: 장비 20대를 4묶음(5대씩)으로 나눠, 묶음마다 학습에 쓰지 않은 장비로만 평가. AUC 0.5는 무작위, 1.0은 완벽.
- **비교**: 누적 카운터 3개(`ETCHSOURCEUSAGE`, `ETCHAUXSOURCETIMER`, `ETCHAUX2SOURCETIMER`)는 노화 효과일 수 있어 넣은 경우와 뺀 경우를 비교한다.

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
TTF = [
    "TTF_FlowCool Pressure Dropped Below Limit",
    "TTF_Flowcool Pressure Too High Check Flowcool Pump",
    "TTF_Flowcool leak",
]
COUNTERS = ["ETCHSOURCEUSAGE", "ETCHAUXSOURCETIMER", "ETCHAUX2SOURCETIMER"]
SENSORS = [
    "IONGAUGEPRESSURE", "ETCHBEAMVOLTAGE", "ETCHBEAMCURRENT",
    "ETCHSUPPRESSORVOLTAGE", "ETCHSUPPRESSORCURRENT",
    "FLOWCOOLFLOWRATE", "FLOWCOOLPRESSURE",
    "ETCHGASCHANNEL1READBACK", "ETCHPBNGASREADBACK",
    "FIXTURETILTANGLE", "ROTATIONSPEED", "ACTUALROTATIONANGLE",
    *COUNTERS, "ACTUALSTEPDURATION",
]
TRAIN_SAMPLE = 300_000
SEED = 0

def show(fig):
    fig.update_layout(template="plotly_white", font=dict(family="Malgun Gothic, sans-serif"))
    fig.show()

## 2. 데이터 불러오기

In [2]:
frames = []
for f in sorted(glob.glob(str(ROOT / "data/processed/minute/*.parquet"))):
    d = pd.read_parquet(f, columns=["Tool", *TTF, *[f"{s}_mean" for s in SENSORS]])
    d["y"] = (d[TTF] <= 86400).any(axis=1).astype("int8")
    frames.append(d.drop(columns=TTF))
df = pd.concat(frames, ignore_index=True)
tools = sorted(df["Tool"].unique())

pd.DataFrame({
    "전체 분": [f"{len(df):,}"],
    "장비 수": [len(tools)],
    "고장 전 분": [f"{int(df['y'].sum()):,}"],
    "고장 전 비율": [f"{df['y'].mean():.1%}"],
}, index=["1분 요약"])

,전체 분,장비 수,고장 전 분,고장 전 비율
1분 요약,"5,777,042",20,"448,711",7.8%


## 3. 장비 4묶음 나누기

장비 이름을 무작위(고정 시드)로 섞어 5대씩 4묶음을 만든다. 묶음 하나가 검증, 나머지 15대가 학습이다.

In [3]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(tools))
folds = [[tools[i] for i in perm[k::4]] for k in range(4)]
pd.DataFrame({"검증 장비": [", ".join(sorted(f)) for f in folds]}, index=[f"묶음 {k+1}" for k in range(4)])

,검증 장비
묶음 1,"03M01, 04M02, 06M01, 07M02, 08M01"
묶음 2,"03M02, 05M01, 05M02, 09M01, 10M02"
묶음 3,"01M01, 01M02, 02M02, 04M01, 10M01"
묶음 4,"02M01, 06M02, 07M01, 08M02, 09M02"


## 4. 학습과 평가

In [1]:
def run(feats):
    cols = [f"{s}_mean" for s in feats]
    aucs, per_tool = [], {}
    for k, test_tools in enumerate(folds):
        is_test = df["Tool"].isin(test_tools)
        train = df[~is_test & (df["y"] == 0)]
        train = train.sample(min(TRAIN_SAMPLE, len(train)), random_state=k)
        med = train[cols].median()
        model = IsolationForest(n_estimators=200, max_samples=256, random_state=k, n_jobs=-1)
        model.fit(train[cols].fillna(med))
        test = df[is_test]
        score = -model.score_samples(test[cols].fillna(med))
        aucs.append(roc_auc_score(test["y"], score))
        for t in test_tools:
            m = (test["Tool"] == t).to_numpy()
            if test.loc[m, "y"].nunique() == 2:
                per_tool[t] = roc_auc_score(test.loc[m, "y"], score[m])
    return aucs, pd.Series(per_tool).sort_index()

res = {
    "카운터 포함 (16개)": run(SENSORS),
    "카운터 제외 (13개)": run([s for s in SENSORS if s not in COUNTERS]),
}

NameError: name 'SENSORS' is not defined

### 4-1. 묶음별 AUC

In [5]:
rows = []
for name, (aucs, per_tool) in res.items():
    rows.append({
        "입력": name,
        **{f"묶음 {k+1}": round(a, 3) for k, a in enumerate(aucs)},
        "평균": round(float(np.mean(aucs)), 3),
        "장비별 중앙값": round(float(per_tool.median()), 3),
        "AUC>0.5 장비": f"{int((per_tool > .5).sum())}/{len(per_tool)}",
    })
pd.DataFrame(rows).set_index("입력")

,묶음 1,묶음 2,묶음 3,묶음 4,평균,장비별 중앙값,AUC>0.5 장비
입력,,,,,,,
카운터 포함 (16개),0.576,0.501,0.612,0.579,0.567,0.539,15/20
카운터 제외 (13개),0.593,0.522,0.614,0.559,0.572,0.535,18/20


### 4-2. 장비별 AUC

In [6]:
order = res["카운터 제외 (13개)"][1].sort_values().index
fig = go.Figure()
for name, (_, per_tool) in res.items():
    fig.add_bar(x=list(order), y=[per_tool[t] for t in order], name=name)
fig.add_hline(y=0.5, line_dash="dash", annotation_text="무작위(0.5)")
fig.update_layout(title="장비별 AUC (학습에 쓰지 않은 장비)", xaxis_title="장비", yaxis_title="AUC",
                  barmode="group", yaxis_range=[0.35, 0.75])
show(fig)

## 5. 해석과 한계

- 평균 AUC는 약 0.57로, 무작위(0.5)보다 조금 높은 정도다. 센서 값이 "정상에서 얼마나 벗어났는가"만으로는 고장 24시간 전을 거의 구분하지 못한다.
- 카운터를 빼도 결과가 비슷하다(0.567 → 0.572). 이 수치는 카운터가 만든 착시가 아니다. 다만 신호가 약한 것이다.
- 장비별 편차가 크다. 07M01·07M02는 0.5 근처이고 일부 장비만 0.6 이상이다.
- 한계: 레시피·단계(recipe, recipe_step)별로 정상 값이 다른데 섞어서 학습했다. 같은 센서라도 공정 단계에 따라 값이 달라 이상 탐지가 흐려질 수 있다.
- 다음 후보: 레시피별로 나눠 학습, 최근 1시간 변화량 같은 창 통계 추가, 지도학습(랜덤포레스트·LightGBM)과 비교.